In [ ]:
from pathlib import Path
import pandas as pd
import optuna

import sys
sys.path.append("../src")

from crop_model_calibration_and_run import *

## Calibrated crop parameters

### MM

In [ ]:
best_params_mm_ythr1pct_0p3_var = {
    "HI0": 0.4922834892179487,
    "WP": 13.345314401231967,
    "CDC": 0.0061176731246107546,
    "Zmax": 0.3528006578851538
}

### TH

In [ ]:
best_params_th_ythr1pct_left_0p5_right_0p1_var = {
    "HI0": 0.2869818659451036,
    "WP": 16.126485507440485,
    "CDC": 0.005707862391921852,
    "CCx": 0.5921401480551517
}

### KH

In [ ]:
best_params_kh_ythr1pct_left_0p6_var = {
    "HI0": 0.2504944986835608,
    "WP": 13.582223320607511,
    "CDC": 0.006295835055926347,
    "CCx": 0.5460006099128839
}

### VN

In [ ]:
best_params_vn_ythr1pct_0p3_var = {
    "WP": 17.418588210018783,
    "HI0": 0.3121568657831974,
    "CDC": 0.004910466564567102,
    "CCx": 0.7197509295595008
}

### PH

In [ ]:
best_params_ph_ythr1pct_0901_0p3_var = {
    "HI0": 0.34998383959959484,
    "WP": 14.718846809485495,
    "CDC": 0.006118362574467186,
    "CCx": 0.6659716564367263
}

### ID

In [ ]:
best_params_id_ythr1pct_0201_0p3_var = {
    'HI0': 0.46768048656672784,
    'WP': 13.737620707189407,
    'CDC': 0.004744348899485139,
    'CCx': 0.6679298810483794
}

## Run
One country/case at one time
### ID/KH/MM/TH/VN (plant and harvest in the same calendar year)

In [ ]:
dict_planting_data = {
    "ID": "02/01",
    "KH": "05/01",
    "MM": "05/01",
    "PH": "09/01",
    "TH": "05/01",
    "VN": "05/01"
}

dict_best_parameters = {
    "ID": best_params_id_ythr1pct_0201_0p3_var,
    "KH": best_params_kh_ythr1pct_left_0p6_var,
    "MM": best_params_mm_ythr1pct_0p3_var,
    "PH": best_params_ph_ythr1pct_0901_0p3_var,
    "TH": best_params_th_ythr1pct_left_0p5_right_0p1_var,
    "VN": best_params_vn_ythr1pct_0p3_var
}

In [ ]:
country_code = "ID"

gid_harvest_area_df = pd.read_csv(f"../../Results/1a_CAs/HAs_{country_code}.csv")
gid_harvest_area_df = gid_harvest_area_df[['GID', 'rice']]
gid_harvest_area_df.columns = ["GID", "harvest_area_ha"]

observed_yield_df = pd.read_csv(
    f"../../Results/3.1_DetrendedYield/DetrendedYield_{country_code}_rice.csv"
)
observed_yield_df = observed_yield_df[["Year", "DetrendQuadratic"]]
observed_yield_df.columns = ["year", "yield_obs_t_ha"]

gid_harvest_area_selected_df = get_gid_ha_by_coverage(
    gid_harvest_area_df,
    threshold=0.9
)

weather_folder_path = (
    Path("../../Data/AquaCropInputs/DailyClimate") / country_code
)

gid_list = gid_harvest_area_selected_df["GID"].tolist()
valid_gids, weather_dfs = get_weather_dfs(gid_list, weather_folder_path)

gid_harvest_area_valid_df = gid_harvest_area_selected_df[
    gid_harvest_area_selected_df["GID"].isin(valid_gids)
    ]

simulation_results_df, comparison_df = simulate_country_yield(
    gid_harvest_area_valid_df,
    weather_dfs,
    observed_yield_df,
    best_crop_parameters=dict_best_parameters[country_code],
    crop_name="PaddyRiceGDD",
    planting_date=dict_planting_data[country_code],
    sim_start_time="1991/01/01",
    sim_end_time="2020/12/31",
    soil_type="ClayLoam",
    yield_threshold=0.01*observed_yield_df["yield_obs_t_ha"].mean(),
)

In [ ]:
simulation_results_df["growing_days"] = (
    pd.to_datetime(simulation_results_df["harvest_date"], format="%Y-%m-%d")
    - pd.to_datetime(
        simulation_results_df["harvest_date"].dt.year.astype(str)
            .add("/")
            .add(simulation_results_df["planting_date"]),
        format="%Y/%m/%d"
    )
).dt.days

print(f"Min growing days: {simulation_results_df["growing_days"].min()}")
print(f"Max growing days: {simulation_results_df["growing_days"].max()}")

simulation_results_df.head()

### PH (one season spanning two calendar years, harvest in the 2nd calendar year)

In [ ]:
gid_harvest_area_df = pd.read_csv("../../Results/1a_CAs/HAs_PH.csv")
gid_harvest_area_df = gid_harvest_area_df[['GID', 'rice']]
gid_harvest_area_df.columns = ["GID", "harvest_area_ha"]

observed_yield_df = pd.read_csv(
    "../../Results/3.1_DetrendedYield/DetrendedYield_PH_rice.csv"
)
observed_yield_df = observed_yield_df[["Year", "DetrendQuadratic"]]
observed_yield_df.columns = ["year", "yield_obs_t_ha"]

gid_harvest_area_selected_df = get_gid_ha_by_coverage(
    gid_harvest_area_df,
    threshold=0.9
)

weather_folder_path = (
    Path("../../Data/AquaCropInputs/DailyClimate") / "PH"
)

gid_list = gid_harvest_area_selected_df["GID"].tolist()
valid_gids, weather_dfs = get_weather_dfs(gid_list, weather_folder_path)

gid_harvest_area_valid_df = gid_harvest_area_selected_df[
    gid_harvest_area_selected_df["GID"].isin(valid_gids)
    ]

simulation_results_df, comparison_df = simulate_country_yield(
    gid_harvest_area_valid_df,
    weather_dfs,
    observed_yield_df,
    best_crop_parameters=best_params_ph_ythr1pct_0901_0p3_var,
    crop_name="PaddyRiceGDD",
    planting_date="09/01",
    user_harvest_year=np.arange(1992, 2021),
    sim_start_time="1991/01/01",
    sim_end_time="2020/12/31",
    soil_type="ClayLoam",
    yield_threshold=0.01*observed_yield_df["yield_obs_t_ha"].mean(),
)

In [ ]:
simulation_results_df["growing_days"] = (
    pd.to_datetime(simulation_results_df["harvest_date"], format="%Y-%m-%d")
    - pd.to_datetime(
        (simulation_results_df["year"]-1).astype(str)
            .add("/")
            .add(simulation_results_df["planting_date"]),
        format="%Y/%m/%d"
    )
).dt.days

print(f"Min growing days: {simulation_results_df["growing_days"].min()}")
print(f"Max growing days: {simulation_results_df["growing_days"].max()}")

simulation_results_df.head()

## Check and save files

In [ ]:
grid_yield_output_path = Path(
    "../../Results/3.2_SimulatedYield/"
    "aquacrop_rice_grid_yield_id_ythr1pct_0201_0p3_var_260715.csv"
)

country_ensemble_yield_output_path = Path(
    "../../Results/3.2_SimulatedYield/"
    "aquacrop_rice_country_ensemble_yield_"
    "id_ythr1pct_0201_0p3_var_260715.csv"
)

In [ ]:
comparison_df.head()

In [ ]:
simulation_results_df = simulation_results_df.round(
    {
        "yield_sim_t_ha": 4,
        "harvest_area_ha": 2,
    }
)

comparison_df = comparison_df.round(
    {
        "yield_sim_t_ha": 2,
        "yield_obs_t_ha": 2,
    }
)

simulation_results_df.to_csv(grid_yield_output_path, index=False)
comparison_df.to_csv(country_ensemble_yield_output_path, index=False)